# 010 — Cómo leer papers, benchmarks y claims de IA

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Lectura en tres pasadas (Keshav):** (1) abstract+figuras+conclusión, 5 min, ¿sigo?;
(2) lectura completa anotando supuestos, 1 h; (3) reconstrucción del experimento — la única
pasada que habilita a citar el paper como evidencia.

**Un benchmark es una muestra congelada de la tarea, no la tarea.** La cadena
capacidad → tarea idealizada → dataset → split → métrica puede romperse por:
contaminación (el test estaba en el train — crítico con LLMs entrenados sobre la web),
sobreajuste comunitario al leaderboard (Goodhart), atajos espurios, saturación, y métricas
agregadas que ocultan subgrupos.

**Checklist mínima ante un claim:** ¿comparación justa (mismo cómputo y búsqueda de
hiperparámetros)? ¿baselines fuertes? ¿cuántas semillas (media ± dispersión)? ¿población
definida? ¿test limpio de leakage y temporalmente posterior? ¿código/datos disponibles?
¿conflictos de interés? ¿evaluación fuera de distribución?

Caso trabajado en la teoría: "AUC 0.94 vs radiólogos 0.87" se desinfla al detectar split
por imagen (no por paciente → leakage), un solo hospital y comparación asimétrica — el
patrón de fallo modal catalogado por Kapoor & Narayanan (2023).

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Críticas típicas aquí: (4) población — ¿qué abogados, con cuánto tiempo y
contexto?; (5) test limpio — ¿los contratos de test son de despachos/plantillas no vistos?;
(1) comparación justa — ¿los humanos tenían las mismas instrucciones y definición de
"riesgo"? Claim honesto: "en cláusulas de contratos de la plantilla P anotadas por nuestro
equipo, el modelo alcanza F1 0.91 frente a 0.84 de abogados sin contexto con 30 s por
cláusula; sin validación en despachos externos".

**Ejercicio 2.** Cláusulas del mismo contrato comparten plantilla, estilo y hasta texto:
con split por cláusula, el modelo ve en train "casi el test". Es el análogo exacto del
split por imagen con varias imágenes del mismo paciente. Split correcto: por contrato (o
por despacho/plantilla si se quiere medir generalización real).

**Ejercicio 3.** B es más informativa: cuantifica la incertidumbre. Si A tuviera σ≈0.6, su
rango ±2σ sería [93.1, 95.5], que solapa de lleno con B (93.8±0.6 → [92.6, 95.0]): no se
puede concluir A > B con una corrida. La décima de diferencia está dentro del ruido.

**Ejercicio 4.** Versión de venta: usa solo el resultado central. Versión con checklist:
añade semilla, condición sintética y el contenido de `limitations`. La diferencia entre
ambas frases es exactamente lo que la clase enseña a detectar en papers ajenos.

In [ ]:
result = run_lab("evaluation", seed=10)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 3 — décimas dentro del ruido
media_A, sigma = 94.3, 0.6
rango_A = (media_A - 2 * sigma, media_A + 2 * sigma)
media_B, sigma_B = 93.8, 0.6
rango_B = (media_B - 2 * sigma_B, media_B + 2 * sigma_B)
print("rango A:", rango_A, "| rango B:", rango_B)
solapan = rango_A[0] <= rango_B[1] and rango_B[0] <= rango_A[1]
print("solapan:", solapan, "→ no se puede afirmar A > B con 1 corrida")
assert solapan

In [ ]:
# Ejercicio 4 — dos abstracts del mismo resultado
r = run_lab("evaluation", seed=1)
venta = f"El método {r['kind']} demuestra resultados sólidos en evaluación."
honesto = (
    f"Con seed=1 en un escenario sintético, '{r['kind']}' produce "
    f"{len(r['evidence'])} evidencias inspeccionables; limitaciones: {r['limitations']}"
)
print("VENTA:  ", venta)
print("HONESTO:", honesto)

## Reflexión

1. Aplica la pasada 1 de Keshav al "paper" implícito en el JSON del laboratorio: con solo
   `kind`, `evidence` y `limitations`, ¿qué decidirías — leer más o descartar — y qué
   pregunta de la checklist responderías primero?
2. ¿Por qué la contaminación de benchmarks es estructuralmente peor con LLMs que con los
   clasificadores de 2015? ¿Qué diseño de evaluación la mitiga?
3. Elige un leaderboard público que conozcas y argumenta: ¿qué eslabón de la cadena
   capacidad→métrica es el más débil en ese caso concreto?